In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm

from gemini_integration import load_api_keys, init_gemini, call_gemini

RAW_RESULTS_CSV = "./evaluation/haifa_rag_strategies_raw_answers.csv"
OUT_SCORED_CSV = "./evaluation/haifa_rag_strategies_scored.csv"

api_keys = load_api_keys("api_keys.json")
gemini_model = init_gemini(api_keys, model_name="gemini-2.5-flash")

df = pd.read_csv(RAW_RESULTS_CSV, encoding="utf-8")
print("Rows:", len(df))
df.head()


In [ ]:
# LLM-as-a-judge

def build_judge_prompt(question: str, gold_answer: str, rag_answer: str) -> str:
    """
    Gemini gets a question, a correct answer, and a RAG answer,
    and returns a score for the RAG answer.
    """
    return f"""
אתה בוחן איכות תשובות.

קיבלת:
שאלה:
\"\"\"{question}\"\"\"

תשובה נכונה (מתוך המסמך הרשמי):
\"\"\"{gold_answer}\"\"\"

תשובה שנוצרה על ידי המערכת (RAG):
\"\"\"{rag_answer}\"\"\"

הערך את התשובה של ה-RAG ביחס לתשובה הנכונה לפי המדדים הבאים (ציון 0 עד 1):

1. correctness - עד כמה התשובה נכונה עובדתית?
2. faithfulness - עד כמה התשובה נאמנה לתשובה הנכונה ולא ממציאה מידע?
3. completeness - עד כמה התשובה מכסה את החלקים החשובים בתשובה הנכונה?
4. conciseness - עד כמה התשובה ברורה ולא מיותרת?
5. overall - ציון כולל כללי.

החזר JSON בלבד בפורמט:
{{
  "correctness": <float>,
  "faithfulness": <float>,
  "completeness": <float>,
  "conciseness": <float>,
  "overall": <float>
}}
"""


def judge_answer(question: str, gold_answer: str, rag_answer: str) -> dict:
    prompt = build_judge_prompt(question, gold_answer, rag_answer)
    resp = call_gemini(gemini_model, prompt)
    try:
        data = json.loads(resp)
        return {
            "correctness": float(data.get("correctness", 0.0)),
            "faithfulness": float(data.get("faithfulness", 0.0)),
            "completeness": float(data.get("completeness", 0.0)),
            "conciseness": float(data.get("conciseness", 0.0)),
            "overall": float(data.get("overall", 0.0)),
        }
    except Exception as e:
        print("Parsing error:", e)
        return {
            "correctness": 0.0,
            "faithfulness": 0.0,
            "completeness": 0.0,
            "conciseness": 0.0,
            "overall": 0.0,
        }


In [ ]:
# Score all answers

scores = {
    "correctness": [],
    "faithfulness": [],
    "completeness": [],
    "conciseness": [],
    "overall": [],
}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row["question"]
    gold = row["gold_answer"]
    rag = row["rag_answer"]

    s = judge_answer(q, gold, rag)
    for k in scores.keys():
        scores[k].append(s[k])

for k, vals in scores.items():
    df[k] = vals

df.head()


In [ ]:
# Save scored results

df.to_csv(OUT_SCORED_CSV, index=False, encoding="utf-8")
print("Saved scored results to:", OUT_SCORED_CSV)
